<a href="https://colab.research.google.com/github/chindarkarsejal800-dot/CSL701-08_Machine-Learning-Lab/blob/main/ML_exp_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Student Name : Sejal Ravindra Chindarkar
Batch :cs1
Roll Number :cs08



### **Aim**

To implement graph-based clustering using Minimum Spanning Tree (MST) on the Breast Cancer Wisconsin (Diagnostic) dataset in Python and visualize the resulting clusters.

## Objectives

1. To understand the concept of graph-based clustering and Minimum Spanning Trees (MST).
2. To load and preprocess the Breast Cancer Wisconsin dataset (scaling features and handling metadata).
3. To construct a distance graph from tumor feature representations using Euclidean distance.
4. To compute the Minimum Spanning Tree using Prim's or Kruskal's algorithm (or via scipy/networkx).
5. To form $K=2$ clusters (Malignant vs. Benign) by removing the single ($K - 1$) largest edge weight from the MST.
6. To evaluate the performance of the MST-based clustering model using evaluation metrics.

# Relevant Theory

###Graph-Based Clustering & Minimum Spanning Tree (MST)
Graph-based clustering represents data points as nodes in a graph, with edges representing relationships or distance metrics between individual samples.
* **Complete Graph Construction**: Given a dataset $X = \{x_1, x_2, \dots, x_n\}$,
each tissue sample is treated as a vertex. An edge weight $w(i, j)$ corresponds to the distance between sample $i$ and sample $j$ (typically Euclidean distance on scaled features):$$w(i, j) = \Vert{}x_i - x_j\Vert{}_2$$
* **Minimum Spanning Tree (MST)**: An MST is a subset of edges connecting all vertices in the graph without forming any cycles, while minimizing the total edge weight sum.
* **MST-Based Clustering Method**:
  1. Construct a fully connected graph with pairwise edge weights across standardized features.
  2. Compute the Minimum Spanning Tree (MST).
  3. To partition the data into $K$ distinct clusters (such as Malignant and Benign groups), identify and remove the $K - 1$ largest (most expensive) edges from the MST.4. The resulting disconnected subgraphs represent the separate data clusters.
### Dataset Information
Get the Breast Cancer Wisconsin (Diagnostic) dataset from Kaggle or via scikit-learn (load_breast_cancer).
The dataset contains key cell nuclei features computed from digitized images of fine needle aspirates (FNA) of breast masses:
  * Total Instances: 569 samples
  * Target Classes: Malignant (M) and Benign (B)* Feature Attributes: 30 numerical features (e.g., mean radius, mean texture, mean perimeter, mean area, mean smoothness)

For 2D visualization, two key features (such as mean radius and mean texture) will be selected.

# Task 1: Import Libraries and Load Dataset

* Import required libraries (numpy, pandas, matplotlib, scipy, networkx, sklearn).

* Load the Breast Cancer dataset using below link from Kaggle.

  https://www.kaggle.com/datasets/utkarshx27/breast-cancer-wisconsin-diagnostic-dataset

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

from scipy.spatial.distance import pdist, squareform
from scipy.sparse.csgraph import minimum_spanning_tree

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics import normalized_mutual_info_score

# Task 2: Explore the Dataset

* Display missing value checks, summary statistics, and feature dimensions.

* Check the diagnosis class distribution (Malignant vs. Benign).

In [ ]:
print("\nColumn Names:")
print(df.columns)

In [ ]:
print("\nData Types:")
print(df.dtypes)

In [ ]:
print("\nMissing Values:")
print(df.isnull().sum())

In [ ]:
print("\nSummary Statistics:")
print(df.describe())

In [ ]:
print("\nDiagnosis Distribution:")
print(df["y"].value_counts())

In [ ]:
plt.figure(figsize=(6, 5))

sns.countplot(x="y", data=df)

plt.title("Diagnosis Class Distribution")
plt.xlabel("Diagnosis")
plt.ylabel("Number of Samples")

plt.show()

# Task 3: Preprocess and Scale Features

* Select numerical features and drop non-informative identifiers (such as id).

* Apply StandardScaler to normalize feature magnitudes prior to graph distance calculation.

In [ ]:
# Remove identifier column
df_clean = df.drop("Unnamed: 0", axis=1)

# Convert target values
# B = 0 (Benign)
# M = 1 (Malignant)

df_clean["y"] = df_clean["y"].map({
    "B": 0,
    "M": 1
})

# Separate features and target
X = df_clean.drop("y", axis=1)
y = df_clean["y"]

print("Feature Shape:", X.shape)
print("Target Shape:", y.shape)

In [ ]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

print("Scaled Feature Shape:", X_scaled.shape)

print("\nFirst 5 scaled samples:")
print(X_scaled[:5])

# Task 4: Construct Pairwise Distance Matrix & Graph

* Compute pairwise Euclidean distance matrix across all scaled samples using scipy.spatial.distance.pdist or cdist.

* Represent the dataset as a weighted complete graph.

In [ ]:
# Calculate pairwise Euclidean distances
distance_vector = pdist(
    X_scaled,
    metric="euclidean"
)

# Convert into square matrix
distance_matrix = squareform(distance_vector)

print("Distance Matrix Shape:",
      distance_matrix.shape)

print("\nFirst 5 x 5 Distance Matrix:")
print(distance_matrix[:5, :5])

NameError: name 'X_scaled' is not defined

In [ ]:
G = nx.Graph()

# Add nodes
G.add_nodes_from(range(len(X_scaled)))

# Add weighted edges
for i in range(len(X_scaled)):
    for j in range(i + 1, len(X_scaled)):
        G.add_edge(
            i,
            j,
            weight=distance_matrix[i, j]
        )

print("\nNumber of Nodes:",
      G.number_of_nodes())

print("Number of Edges:",
      G.number_of_edges())

# Task 5: Compute the Minimum Spanning Tree (MST)

* Compute the MST using scipy.sparse.csgraph.minimum_spanning_tree or networkx.minimum_spanning_tree.

* Extract the resulting graph connections and weights.

In [ ]:
# Compute MST
mst_sparse = minimum_spanning_tree(
    distance_matrix
)

# Convert to NetworkX graph
mst = nx.from_scipy_sparse_array(
    mst_sparse
)

print("MST Nodes:",
      mst.number_of_nodes())

print("MST Edges:",
      mst.number_of_edges())

In [ ]:
mst_edges = []

for u, v, data in mst.edges(data=True):

    mst_edges.append(
        (u, v, data["weight"])
    )

mst_edges_df = pd.DataFrame(
    mst_edges,
    columns=[
        "Node1",
        "Node2",
        "Weight"
    ]
)

print(mst_edges_df.head())

print("\nTotal MST Edges:",
      len(mst_edges_df))

# Task 6: Implement Graph-Based Clustering ($K=2$)

* Sort the edges of the MST in descending order by edge weight.
* Remove the largest $K - 1$ edge (1 largest edge for 2 clusters) to split the MST into 2 connected components.
* Assign final cluster labels based on the resulting component indices.


In [ ]:
mst_edges_sorted = sorted(
    mst_edges,
    key=lambda x: x[2],
    reverse=True
)

print("10 Largest MST Edges:")

for edge in mst_edges_sorted[:10]:
    print(edge)

In [ ]:
# Copy MST
mst_cluster = mst.copy()

# Get largest edge
largest_edge = mst_edges_sorted[0]

node1 = int(largest_edge[0])
node2 = int(largest_edge[1])
weight = largest_edge[2]

print("\nLargest Edge:")
print("Node 1:", node1)
print("Node 2:", node2)
print("Weight:", weight)

# Remove largest edge
mst_cluster.remove_edge(
    node1,
    node2
)

In [ ]:
components = list(
    nx.connected_components(
        mst_cluster
    )
)

print("\nNumber of Clusters:",
      len(components))

print("\nCluster Sizes:")

for i, component in enumerate(components):

    print(
        "Cluster",
        i,
        ":",
        len(component),
        "samples"
    )

In [ ]:
cluster_labels = np.zeros(
    len(X_scaled),
    dtype=int
)

for cluster_id, component in enumerate(
    components
):

    for node in component:
        cluster_labels[node] = cluster_id

print("\nCluster Labels:")
print(cluster_labels)

print("\nCluster Distribution:")
print(
    pd.Series(
        cluster_labels
    ).value_counts()
)

# Task 7: valuate MST Clustering Performance

Evaluate cluster assignment against ground truth labels using:

  *  Silhouette Score

  *  Adjusted Rand Index (ARI)

  *  Normalized Mutual Information (NMI)

In [ ]:
silhouette = silhouette_score(
    X_scaled,
    cluster_labels
)

print("Silhouette Score:",
      silhouette)

In [ ]:
ari = adjusted_rand_score(
    y,
    cluster_labels
)

print("Adjusted Rand Index:",
      ari)

In [ ]:
nmi = normalized_mutual_info_score(
    y,
    cluster_labels
)

print("Normalized Mutual Information:",
      nmi)

In [ ]:
print("======================================")
print("       MST CLUSTERING RESULTS")
print("======================================")

print(
    "Silhouette Score :",
    round(silhouette, 4)
)

print(
    "Adjusted Rand Index (ARI) :",
    round(ari, 4)
)

print(
    "Normalized Mutual Information (NMI) :",
    round(nmi, 4)
)

# Task 8: Visualize MST and Graph-Based Clusters

* Select two representative features (e.g., mean radius vs. mean texture) for 2D plotting.
 * Plot 1: Unclustered standardized data points.
* Plot 2: Minimum Spanning Tree (MST) edge connections overlaid on the data.
* Plot 3: Final clusters formed after cutting the $K - 1$ largest edge(s).

In [ ]:
feature_x = "x.radius_mean"
feature_y = "x.texture_mean"

plt.figure(figsize=(8, 6))

plt.scatter(
    df[feature_x],
    df[feature_y],
    s=40
)

plt.xlabel("Mean Radius")
plt.ylabel("Mean Texture")
plt.title("Unclustered Breast Cancer Dataset")

plt.grid()

plt.show()

In [ ]:
plt.figure(figsize=(10, 8))

# Draw MST edges
for u, v, data in mst.edges(data=True):

    plt.plot(
        [
            df.iloc[u][feature_x],
            df.iloc[v][feature_x]
        ],

        [
            df.iloc[u][feature_y],
            df.iloc[v][feature_y]
        ],

        linewidth=0.5,
        alpha=0.5
    )

# Draw data points
plt.scatter(
    df[feature_x],
    df[feature_y],
    s=25
)

plt.xlabel("Mean Radius")
plt.ylabel("Mean Texture")
plt.title("Minimum Spanning Tree (MST)")

plt.grid()

plt.show()

In [ ]:
plt.figure(figsize=(9, 7))

sns.scatterplot(
    x=df[feature_x],
    y=df[feature_y],
    hue=cluster_labels,
    palette="Set1",
    s=60
)

plt.xlabel("Mean Radius")
plt.ylabel("Mean Texture")

plt.title(
    "MST-Based Clustering (K=2)"
)

plt.legend(title="Cluster")

plt.grid()

plt.show()

# Conclusion

The experiment successfully demonstrated the implementation of Graph-Based Clustering using a Minimum Spanning Tree (MST) on the Breast Cancer Wisconsin (Diagnostic) dataset. The dataset was preprocessed, cleaned, and standardized before computing pairwise distances. The MST was generated, and by removing the largest edge(s), clusters representing tumor categories were obtained, evaluated using clustering metrics, and visualized.